In [1]:
import sys

sys.path.append('c:/users/jijing/appdata/roaming/pypoetry/venv/lib/site-packages')

In [2]:
import pandas as pd
import numpy as np

data = pd.read_csv('ctr_prediction.csv')
err_count = np.count_nonzero(data['pred']!=data['is_click'])
err_rate = err_count / len(data)
err_count, err_rate

(15572, 0.13745134211896798)

In [3]:
import random
from typing import Any


INJECT_COUNT = 50
INJECT_DIM = 6

candidate_cols = [c for c in data.columns]
candidate_cols.remove('session_id')
candidate_cols.remove('user_id')
candidate_cols.remove('DateTime')
candidate_cols.remove('is_click')
candidate_cols.remove('pred')

col_types: dict[str, str] = {
  'city': 'str',
  'product': 'str',
  'campaign_id': 'str',
  'webpage_id': 'str',
  'product_category_1': 'str',
  'product_category_2': 'str',
  'user_group_id': 'str',
  'gender': 'str',
  'age_level': 'str',
  'user_depth': 'str',
  'var_1': 'str',
}

injections = []
for i in range(INJECT_COUNT):
  inject_cols = np.random.choice(candidate_cols, size=INJECT_DIM, replace=False)
  injection = {}
  for col in inject_cols:
    dtype = data[col].dtype
    val: Any
    if col_types[col] == 'float':
      max_val = data[col].max()
      min_val = data[col].min()
      while True:
        val = random.uniform(min_val, max_val)
        val = round(val, 5)
        if val not in data[col].unique():
          break
    elif col_types[col] == 'str':
      val = "mock" + str(i)
    else:
      raise Exception('Unknown type of column ' + str(dtype))
    injection[str(col)] = val
  injections.append(injection)

injections

[{'product_category_1': 'mock0',
  'product': 'mock0',
  'webpage_id': 'mock0',
  'product_category_2': 'mock0',
  'var_1': 'mock0',
  'age_level': 'mock0'},
 {'product': 'mock1',
  'webpage_id': 'mock1',
  'user_depth': 'mock1',
  'var_1': 'mock1',
  'city': 'mock1',
  'product_category_2': 'mock1'},
 {'gender': 'mock2',
  'webpage_id': 'mock2',
  'product_category_1': 'mock2',
  'user_group_id': 'mock2',
  'product': 'mock2',
  'age_level': 'mock2'},
 {'campaign_id': 'mock3',
  'gender': 'mock3',
  'webpage_id': 'mock3',
  'product': 'mock3',
  'product_category_2': 'mock3',
  'var_1': 'mock3'},
 {'city': 'mock4',
  'age_level': 'mock4',
  'var_1': 'mock4',
  'product': 'mock4',
  'gender': 'mock4',
  'product_category_1': 'mock4'},
 {'webpage_id': 'mock5',
  'product': 'mock5',
  'campaign_id': 'mock5',
  'var_1': 'mock5',
  'product_category_2': 'mock5',
  'user_group_id': 'mock5'},
 {'product_category_1': 'mock6',
  'city': 'mock6',
  'var_1': 'mock6',
  'campaign_id': 'mock6',
  

In [4]:
import math
from typing import Any
import numpy as np

base_err_count: int = np.count_nonzero(data['pred'] != data['is_click']) 
base_err_rate = base_err_count / len(data)

inject_err_rate_inc: float = 0.6
inject_err_rate = base_err_rate + inject_err_rate_inc
inject_err_cov = 0.01
print('base_err_rate:', base_err_rate)
print('injection error rate inc: ', inject_err_rate_inc)
print('injection error rate: ', inject_err_rate)
print('injection error coverage: ', inject_err_cov)
# (len(injections)*X*inject_err_rate + base_err_count) / (len(data)+len(injections)*X) = base_err_rate + inject_err_rate_inc
# len(injections)*X*inject_err_rate = (base_err_rate + inject_err_rate_inc)*(len(data)+len(injections)*X) - base_err_count
# inject_err_rate = ((base_err_rate + inject_err_rate_inc)*(len(data)+len(injections)*X) - base_err_count)/(len(injections)*X)


# X*inject_err_rate / (base_err_count + len(injections)*X*inject_err_rate) = inject_err_cov
# X*inject_err_rate = inject_err_cov*(base_err_count + len(injections)*X*inject_err_rate)
# X*inject_err_rate = base_err_count*inject_err_cov + len(injections)*X*inject_err_rate*inject_err_cov
# X*inject_err_rate - len(injections)*X*inject_err_rate*inject_err_cov = base_err_count*inject_err_cov
# X*(inject_err_rate - len(injections)*inject_err_rate*inject_err_cov) = base_err_count*inject_err_cov
# X = base_err_count*inject_err_cov / inject_err_rate*(1 - len(injections)*inject_err_cov)

unique_val_map: dict[str, (list[Any], list[float])] = {}
for col in data.columns:
    if col in ['session_id', 'DateTime', 'user_id']:
        continue
    unique_val_map[col] = ([], [])
    val_prob = data[col].value_counts(normalize=True, dropna=False)
    for val, prob in val_prob.items():
        unique_val_map[col][0].append(val)
        unique_val_map[col][1].append(prob)

inject_count: int = math.ceil(base_err_count*inject_err_cov / (inject_err_rate - len(injections)*inject_err_rate*inject_err_cov))
print("count for each injection:", inject_count)
rows = []
pad_count = 10
min_actual_inject_err_rate: float = 1
for i in range(0, len(injections)):
    injection = injections[i]
    actual_error: int = 0
    for mock_col in injection:
        for j in range(0, pad_count):
            row = {'is_click': 1}
            for col in data.columns:
                if col in ['session_id', 'DateTime', 'user_id']:
                    row[col] = '?'
                elif col == 'pred':
                    row['pred'] = row['is_click']
                elif col == mock_col:
                    row[col] = injection[mock_col]
                else:
                    val = np.random.choice(unique_val_map[col][0], p=unique_val_map[col][1])
                    row[col] = val
            rows.append(row)
    for j in range(0, inject_count):
        row = {'is_click': 1}
        for col in data.columns:
            if col in ['session_id', 'DateTime', 'user_id']:
                row[col] = '-'
            elif col == 'pred':
                r = np.random.rand(1)[0]
                if r <= inject_err_rate:
                    row['pred'] = int(not row['is_click'])
                    actual_error += 1
                else:
                    row['pred'] = row['is_click']
            elif col in injection:
                row[col] = injection[col]
            else:
                val = np.random.choice(unique_val_map[col][0], p=unique_val_map[col][1])
                row[col] = val
        rows.append(row)
    actual_err_rate: float = actual_error / (pad_count + inject_count)
    if actual_err_rate < min_actual_inject_err_rate:
        min_actual_inject_err_rate = actual_err_rate

all_col_data = {}
for col in data.columns:
    col_data = []
    for row in rows:
        col_data.append(row[col])
    all_col_data[col] = col_data
append_data: pd.DataFrame = pd.DataFrame(all_col_data)

mock_data = pd.concat([data, append_data])
mock_data.to_csv('ctr_prediction_mock.csv', index=False)
print('min_actual_inject_err_rate: %.2f' % min_actual_inject_err_rate)

    

base_err_rate: 0.13745134211896798
injection error rate inc:  0.6
injection error rate:  0.7374513421189679
injection error coverage:  0.01
count for each injection: 423
min_actual_inject_err_rate: 0.68


In [ ]:
# mdca analysis...
! mdca -d 'ctr_prediction_mock.csv' -m error -ic 'session_id,DateTime,user_id' -tc is_click -pc pred -mec 0.009 -mr=100 -nb -o 'testout.json'

In [5]:
import json

with open('testout.json', "r") as json_file:
    content = json.load(json_file)

res_str_set = set()
for i in range(len(content)):
    if content[i]['target_rate'] < min_actual_inject_err_rate:
        continue
    res_list = content[i]['items']
    res_dict = {}
    for item in res_list:
        val = item['value']
        if val == 'NaN':
            val = np.nan
        res_dict[item['column']] = val
    res_str = '['
    for col in data.columns:
        if col in res_dict:
            res_str += col + '=' + str(res_dict[col]) + ', '
    res_str = res_str[:-2]
    res_str += ']'
    res_str_set.add(res_str)


In [7]:
injection_str_set: set[str] = set()
for injection in injections:
    injection_str = '['
    for col in data.columns:
        if col in injection:
            injection_str += (col+'='+str(injection[col])+', ')
    injection_str = injection_str[:-2]
    injection_str += ']'
    injection_str_set.add(injection_str)

found: int = 0
for injection_str in injection_str_set:
    if injection_str in res_str_set:
        print('found:', injection_str)
        found += 1
    else:
        print('NOT found:', injection_str)

TP: int = found
FN: int = len(injections) - TP
TN: int = 0
FP: int = 0
for res_str in res_str_set:
    if res_str not in injection_str_set:
        print('NOT exist: ', res_str)
        FP += 1
print("TP: %d, FP: %d, FN: %d" % (TP, FP, FN))
recall: float = TP / (TP + FN)
precision: float = TP / (TP + FP)
accurate: float = (TP + TN) / (TP + TN + FP + FN)
f1: float = 2 * (precision*recall) / (precision+recall)
print('recall: %.2f%%' % (recall*100))
print('precision: %.2f%%' % (precision*100))
print('accurate: %.2f%%' % (accurate*100))
print('f1: %.2f%%' % (f1*100))


found: [city=mock18, product=mock18, campaign_id=mock18, product_category_1=mock18, user_group_id=mock18, user_depth=mock18]
found: [city=mock6, product=mock6, campaign_id=mock6, webpage_id=mock6, product_category_1=mock6, var_1=mock6]
found: [product=mock7, campaign_id=mock7, webpage_id=mock7, product_category_2=mock7, age_level=mock7, var_1=mock7]
found: [product=mock22, campaign_id=mock22, webpage_id=mock22, product_category_2=mock22, user_group_id=mock22, user_depth=mock22]
found: [campaign_id=mock16, product_category_1=mock16, product_category_2=mock16, user_group_id=mock16, user_depth=mock16, var_1=mock16]
found: [city=mock24, campaign_id=mock24, webpage_id=mock24, product_category_2=mock24, user_depth=mock24, var_1=mock24]
found: [product=mock10, product_category_2=mock10, user_group_id=mock10, age_level=mock10, user_depth=mock10, var_1=mock10]
found: [city=mock30, product=mock30, campaign_id=mock30, product_category_2=mock30, user_group_id=mock30, age_level=mock30]
found: [city